In [1]:
import sys
import pandas as pd

sys.path.append("../src")  # permet d'importer les fichiers de src/
from metrics import GROUPES, evaluer, rapport_par_groupe, probas_vers_predictions, chercher_seuils
train = pd.read_csv("../data/processed/train.csv", dtype={"id": str})
val = pd.read_csv("../data/processed/val.csv", dtype={"id": str})

print("Train :", train.shape, "| Validation :", val.shape)
print("Import de metrics.py : OK")

Train : (12132, 11) | Validation : (2609, 11)
Import de metrics.py : OK


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectoriseur = TfidfVectorizer(
    lowercase=True,         # tout en minuscules
    stop_words="english",   # retire les mots vides : the, of, and...
    ngram_range=(1, 2),     # mots seuls ET paires de mots ("neural network")
    min_df=2,               # ignore les mots présents dans moins de 2 résumés
    max_features=100_000,   # garde au maximum 100 000 mots/paires
    sublinear_tf=True,      # atténue l'effet d'un mot répété 10 fois
)

X_train = vectoriseur.fit_transform(train["text"])  # apprend le vocabulaire SUR LE TRAIN uniquement
X_val = vectoriseur.transform(val["text"])          # réutilise ce vocabulaire sur la validation

print("X_train :", X_train.shape)
print("X_val   :", X_val.shape)

X_train : (12132, 100000)
X_val   : (2609, 100000)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

y_train = train[GROUPES].values  # les 8 colonnes de 0/1
y_val = val[GROUPES].values

modele = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=4.0))
modele.fit(X_train, y_train)
print("Entraînement terminé")

Entraînement terminé


In [4]:
probas_val = modele.predict_proba(X_val)        # probabilités entre 0 et 1, forme (2609, 8)
pred_val = probas_vers_predictions(probas_val)  # seuil 0.5 -> 0 ou 1

scores = evaluer(y_val, pred_val)
for nom, valeur in scores.items():
    print(f"{nom:16s} {valeur:.4f}")
print()
print(rapport_par_groupe(y_val, pred_val))

micro_f1         0.7606
macro_f1         0.7488
hamming_loss     0.0893
subset_accuracy  0.5040

              precision    recall  f1-score   support

          cs       0.84      0.81      0.82      1023
        math       0.84      0.63      0.72       606
     physics       0.91      0.71      0.80       623
       q-bio       0.92      0.66      0.77       389
       q-fin       0.90      0.69      0.78       436
        stat       0.76      0.56      0.65       484
        eess       0.84      0.63      0.72       383
        econ       0.85      0.65      0.74       366

   micro avg       0.85      0.69      0.76      4310
   macro avg       0.86      0.67      0.75      4310
weighted avg       0.85      0.69      0.76      4310
 samples avg       0.83      0.75      0.76      4310



In [5]:
import json

with open("../results/baseline_tfidf_val.json", "w") as f:
    json.dump(scores, f, indent=2)
print("Scores sauvegardés dans results/baseline_tfidf_val.json")

Scores sauvegardés dans results/baseline_tfidf_val.json


In [6]:
seuils = chercher_seuils(y_val, probas_val)
print("Seuils choisis :", dict(zip(GROUPES, seuils.tolist())))
print()

pred_val_seuils = probas_vers_predictions(probas_val, seuils)
scores_seuils = evaluer(y_val, pred_val_seuils)
for nom, valeur in scores_seuils.items():
    print(f"{nom:16s} {valeur:.4f}")
print()
print(rapport_par_groupe(y_val, pred_val_seuils))

with open("../results/baseline_tfidf_val_seuils.json", "w") as f:
    json.dump({"seuils": seuils.tolist(), **scores_seuils}, f, indent=2)
print("Scores sauvegardés dans results/baseline_tfidf_val_seuils.json")

Seuils choisis : {'cs': 0.4, 'math': 0.3, 'physics': 0.3, 'q-bio': 0.25, 'q-fin': 0.3, 'stat': 0.4, 'eess': 0.35, 'econ': 0.25}

micro_f1         0.7915
macro_f1         0.7868
hamming_loss     0.0880
subset_accuracy  0.5029

              precision    recall  f1-score   support

          cs       0.80      0.86      0.83      1023
        math       0.71      0.78      0.74       606
     physics       0.81      0.83      0.82       623
       q-bio       0.84      0.87      0.85       389
       q-fin       0.82      0.81      0.82       436
        stat       0.71      0.66      0.69       484
        eess       0.77      0.77      0.77       383
        econ       0.72      0.85      0.78       366

   micro avg       0.77      0.81      0.79      4310
   macro avg       0.77      0.80      0.79      4310
weighted avg       0.78      0.81      0.79      4310
 samples avg       0.81      0.86      0.80      4310

Scores sauvegardés dans results/baseline_tfidf_val_seuils.json
